This notebook collects all Uber articles from the Guardian and Daily Mail using SERPAPI.

In [ ]:
# Pip installs
!pip3 install beautifulsoup4
!pip3 install requests
!pip3 install serpapi

# Imports
from bs4 import BeautifulSoup
import requests
import serpapi
import json
from datetime import datetime
import dateutil.parser as parser
import math

This block defines a function that gets all of the specified newspapers' stories that include the given keyword in their headline.
It then runs the function for each newspaper, storing the result in their respective raw json files.

In [ ]:
#This dictionary is used by the following function - if paper not recognised [] is returned
# This uses SerpAPI

papers = {
    'guardian': "www.theguardian.com",
    'dailymail' : "www.dailymail.co.uk"
}

# return a list of articles of form
"""
{headline : text, body: text, published: date, updated: date, authors: list}
"""
def get_all_articles(paper, keyword):
  # Check that paper exists
  if paper.lower() not in papers:
    return []
  searches = []
  end = False

  start = 0

  while(not end):


    params = {
      "q": 'allintitle:"' + keyword + '" site:' + papers[paper] + '',
      "location": "London, United Kingdom",
      "hl": "en",
      "gl": "gb",
      "google_domain": "google.com",
      "api_key": "___",
      "start": start,
      "output": 'json'
    }

    search = serpapi.search(params).as_dict()
    # Check if any more results have been fetched
    if(search['search_information']['organic_results_state'] == 'Fully empty'):
      end = True
    else:
      searches.append(search)
      start += 10
      print("Done page " + str(start/10))
  return searches

# fake paper name not to drain SERPAPI requests in case I accidentally hate this.
# this creates two files one for daily mail and one for guardian
for paper_name in list(papers.keys()):
  searchResults = (get_all_articles(paper_name, 'uber'))

  json_object = json.dumps(searchResults, indent=4)
  with open("raw_" + paper_name + ".json", "w") as outfile:
      outfile.write(json_object)
